# Evaluation of predictive maps

In [ ]:
!pip install -q git+https://github.com/hyunwoogu/salience_bias.git

In [ ]:
import json
import pickle
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from pathlib import Path

from salience_bias.analyses.evaluate.api import (
    ConvexMixtureResult,
    evaluate_convex_mixture,
    evaluate_single,
)
from salience_bias.analyses.evaluate.datasets.steer_search import zoom_to_hw

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
path_pred_map0 = Path('/content/drive/MyDrive/data/steer_search/pred_map0')
path_pred_map1 = Path('/content/drive/MyDrive/data/steer_search/pred_map1')
path_images    = Path('/content/drive/MyDrive/data/steer_search/images')

img = Image.open( path_images / 'search_0002_0191.jpg' )

# example fixations (in visual angles)
eye_pos = {"x": [-4.04318796, -7.4481436 , 11.16194781, 15.23417391,  1.39847154],
           "y": [ 4.30528831,  5.27311933,  5.53288994,  5.87903692,  5.13348817]}

with open( path_pred_map0 / 'search_0002_0191.pkl', 'rb' ) as f:
    pred_map0 = pickle.load(f)

with open( path_pred_map1 / 'search_0002_0191.pkl', 'rb' ) as f:
    pred_map1 = pickle.load(f)

In [ ]:
f,ax = plt.subplots(1,4,figsize=[12,3], sharex=True, sharey=True)

# image
extent = [-17.5, 17.5, -17.5, 17.5]
ax[0].set_title('Image')
ax[0].imshow(img, extent=extent)

# image overlayed with scanpath
ax[1].set_title('Image with fixations')
ax[1].imshow( img, extent=extent, alpha=0.5 )
ax[1].scatter( eye_pos['x'], eye_pos['y'])
ax[1].plot( eye_pos['x'], eye_pos['y'], lw=1 )

ax[2].set_title('Predictive map 0')
ax[2].imshow(pred_map0, extent=extent)

ax[3].set_title('Predictive map 1')
ax[3].imshow(pred_map1, extent=extent)

for ifig in range(4):
    ax[ifig].set_xlabel('X (deg)')
ax[0].set_ylabel('Y (deg)')

plt.show()

In [ ]:
gauss_params = {
    "h_nbin": 224,
    "w_nbin": 224,
    "normalize": False,
    "h_range": (+17.5, -17.5),
    "w_range": (-17.5, +17.5),
}

score_uauc = evaluate_single(
    zoom_to_hw(pred_map0),
    eye_pos,
    metric="cc", # correlation
    gauss_params=gauss_params,
)
score_ll = evaluate_single(
    zoom_to_hw(pred_map0),
    eye_pos,
    metric="ll", # log likelihood
    gauss_params=gauss_params,
)
print(f"correlation coefficient = {score_uauc:.4f}")
print(f"log likelihood = {score_ll:.4f}")

In [ ]:
lambdas = np.linspace(0, 1, 40)

res_cc: ConvexMixtureResult = evaluate_convex_mixture(
    zoom_to_hw(pred_map0),
    zoom_to_hw(pred_map1),
    eye_pos,
    metric="cc",
    lambdas=lambdas,
    gauss_params=gauss_params,
)

res_ll = evaluate_convex_mixture(
    zoom_to_hw(pred_map0),
    zoom_to_hw(pred_map1),
    eye_pos,
    metric="ll",
    lambdas=lambdas,
    gauss_params=gauss_params,
)

fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].plot(res_cc.lambdas, res_cc.scores, "-o", ms=3)
ax[0].axvline(res_cc.best_lambda, color="gray", ls="--", label=f"best λ={res_cc.best_lambda:.2f}")
ax[0].set_xlabel("λ (weight on pred_map1)")
ax[0].set_ylabel("Correlation coefficient")
ax[0].set_title("Convex mixture (correlation)")
ax[0].legend()

ax[1].plot(res_ll.lambdas, res_ll.scores, "-o", ms=3, color="darkgreen")
ax[1].axvline(res_ll.best_lambda, color="gray", ls="--", label=f"best λ={res_ll.best_lambda:.2f}")
ax[1].set_xlabel("λ (weight on pred_map1)")
ax[1].set_ylabel("Log likelihood")
ax[1].set_title("Convex mixture (log likelihood)")
ax[1].legend()
plt.tight_layout()
plt.show()

print("cc best:", res_cc.best_lambda, res_cc.best_score)
print("ll best:", res_ll.best_lambda, res_ll.best_score)